## Explotación Analítica del Cubo SECOP

Taller ETL – Cubo SECOP Autor: Jurani Zabala Hernandez

**Objetivo:** Explotar el cubo `secop_dw` (poblado por `Cargue.ipynb`) con consultas SQL de negocio, distintas a las de validación de `Cargue.ipynb`.

## 1. Sesión de Spark con soporte Hive

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SECOP_ConsultasSQL")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

DB_NAME = "secop_dw"
spark.catalog.setCurrentDatabase(DB_NAME)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Sesión Spark inicializada. Base de datos activa: {DB_NAME}")
spark.sql(f"SHOW TABLES IN {DB_NAME}").show(truncate=False)

26/08/24 02:34:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/24 02:34:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/24 02:34:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/08/24 02:35:09 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/08/24 02:35:09 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.6
26/08/24 02:35:09 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


✅ Sesión Spark inicializada. Base de datos activa: secop_dw
+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|secop_dw|dim_categoria      |false      |
|secop_dw|dim_entidad        |false      |
|secop_dw|dim_estado_contrato|false      |
|secop_dw|dim_modalidad      |false      |
|secop_dw|dim_proveedor      |false      |
|secop_dw|dim_tiempo         |false      |
|secop_dw|dim_ubicacion      |false      |
|secop_dw|hecho_contratos    |false      |
+--------+-------------------+-----------+



## 2. Panorama general del cubo

In [3]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_contratos,
    format_number(SUM(valor_contrato), 2) AS valor_total_contratado,
    format_number(AVG(valor_contrato), 2) AS valor_promedio_contrato,
    format_number(SUM(valor_pendiente_pago), 2) AS total_pendiente_pago
FROM {DB_NAME}.hecho_contratos
""").show(truncate=False)

+---------------+----------------------+-----------------------+--------------------+
|total_contratos|valor_total_contratado|valor_promedio_contrato|total_pendiente_pago|
+---------------+----------------------+-----------------------+--------------------+
|18581          |2,336,619,081,251.44  |125,753,139.30         |1,620,542,533,140.00|
+---------------+----------------------+-----------------------+--------------------+



## 3. Evolución mensual del valor contratado (usando `dim_tiempo` en su rol de fecha de firma)

In [5]:
spark.sql(f"""
SELECT t.anio, t.mes, t.nombre_mes, COUNT(*) AS cantidad_contratos, format_number(SUM(h.valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_tiempo t ON h.sk_tiempo_firma = t.sk_tiempo
GROUP BY t.anio, t.mes, t.nombre_mes
ORDER BY t.anio, t.mes
""").show(30, truncate=False)

+----+---+----------+------------------+-----------------+
|anio|mes|nombre_mes|cantidad_contratos|valor_total      |
+----+---+----------+------------------+-----------------+
|2016|1  |January   |1                 |38,640,000.00    |
|2016|7  |July      |1                 |21,948,384.00    |
|2016|8  |August    |2                 |51,180,152.00    |
|2016|9  |September |1                 |80,012,160.00    |
|2016|12 |December  |1                 |127,000,000.00   |
|2017|1  |January   |5                 |351,393,450.00   |
|2017|2  |February  |4                 |194,135,363.00   |
|2017|3  |March     |6                 |187,570,566.67   |
|2017|4  |April     |2                 |49,200,000.00    |
|2017|5  |May       |6                 |106,149,563.00   |
|2017|6  |June      |1                 |6,780,000.00     |
|2017|7  |July      |1                 |56,221,000.00    |
|2017|8  |August    |5                 |288,625,100.00   |
|2017|9  |September |7                 |718,757,949.00  

## 4. Concentración del gasto: top 10 entidades y su % del total

In [9]:
spark.sql(f"""
WITH por_entidad AS (
    SELECT e.nombre_entidad, SUM(h.valor_contrato) AS valor_entidad
    FROM {DB_NAME}.hecho_contratos h
    JOIN {DB_NAME}.dim_entidad e ON h.sk_entidad = e.sk_entidad
    GROUP BY e.nombre_entidad
),
total AS (SELECT SUM(valor_entidad) AS valor_total FROM por_entidad)
SELECT
    p.nombre_entidad,
    format_number(p.valor_entidad, 2) AS valor_entidad,
    format_number(100 * p.valor_entidad / t.valor_total, 2) AS porcentaje_del_total
FROM por_entidad p, total t
ORDER BY p.valor_entidad DESC
LIMIT 10
""").show(truncate=False)

26/08/24 02:42:23 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/08/24 02:42:23 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/24 02:42:23 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


+------------------------------------------------------------------+------------------+--------------------+
|nombre_entidad                                                    |valor_entidad     |porcentaje_del_total|
+------------------------------------------------------------------+------------------+--------------------+
|DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN  |159,042,883,456.00|6.81                |
|INVIAS                                                            |127,885,117,423.34|5.47                |
|JEP                                                               |63,042,798,203.00 |2.70                |
|DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA          |59,508,583,079.00 |2.55                |
|IDRD - ENTIDAD OFICIAL.                                           |52,473,508,828.00 |2.25                |
|UNP                                                               |49,502,462,037.00 |2.12                |
|UARIV-UNIDAD PARA 

## 5. Modalidad de contratación por sector

In [11]:
spark.sql(f"""
SELECT e.sector, m.modalidad_contratacion, COUNT(*) AS cantidad_contratos, format_number(SUM(h.valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_entidad e ON h.sk_entidad = e.sk_entidad
JOIN {DB_NAME}.dim_modalidad m ON h.sk_modalidad = m.sk_modalidad
GROUP BY e.sector, m.modalidad_contratacion
ORDER BY e.sector, valor_total DESC
""").show(30, truncate=False)

+--------------------------------+-----------------------------------------------------------+------------------+-----------------+
|sector                          |modalidad_contratacion                                     |cantidad_contratos|valor_total      |
+--------------------------------+-----------------------------------------------------------+------------------+-----------------+
|Ambiente y Desarrollo Sostenible|Mínima cuantía                                             |41                |883,309,076.13   |
|Ambiente y Desarrollo Sostenible|Contratación Directa (con ofertas)                         |11                |7,544,973,705.00 |
|Ambiente y Desarrollo Sostenible|Contratación directa                                       |733               |30,061,515,138.03|
|Ambiente y Desarrollo Sostenible|Licitación pública                                         |3                 |3,218,489,460.50 |
|Ambiente y Desarrollo Sostenible|Contratación régimen especial (con ofertas

## 6. Participación de proveedores PYME en el valor contratado

In [13]:
spark.sql(f"""
SELECT
    p.es_pyme,
    COUNT(DISTINCT h.sk_proveedor) AS proveedores_distintos,
    COUNT(*) AS cantidad_contratos,
    format_number(SUM(h.valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_proveedor p ON h.sk_proveedor = p.sk_proveedor
GROUP BY p.es_pyme
""").show(truncate=False)

+-------+---------------------+------------------+--------------------+
|es_pyme|proveedores_distintos|cantidad_contratos|valor_total         |
+-------+---------------------+------------------+--------------------+
|No     |15499                |15838             |1,630,416,479,870.37|
|Si     |2502                 |2743              |706,202,601,381.07  |
+-------+---------------------+------------------+--------------------+



## 7. Cumplimiento de pagos: contratos con mayor % pendiente

In [17]:
spark.sql(f"""
SELECT
    e.nombre_entidad, h.id_contrato, format_number(h.valor_contrato,2) as valor_contrato, format_number(h.valor_pagado,2) as valor_pagado, format_number(h.valor_pendiente_pago,2) as valor_pendiente_pago,
    format_number(100 * h.valor_pendiente_pago / NULLIF(h.valor_contrato, 0), 2) AS porcentaje_pendiente
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_entidad e ON h.sk_entidad = e.sk_entidad
WHERE h.valor_pendiente_pago IS NOT NULL AND h.valor_contrato > 0
ORDER BY porcentaje_pendiente DESC
LIMIT 15
""").show(truncate=False)

+---------------------------------------------------------+------------------+----------------+-------------+--------------------+--------------------+
|nombre_entidad                                           |id_contrato       |valor_contrato  |valor_pagado |valor_pendiente_pago|porcentaje_pendiente|
+---------------------------------------------------------+------------------+----------------+-------------+--------------------+--------------------+
|EMPRESA SOCIAL DEL ESTADO DEL MUNICIPIO DE VILLAVICENCIO |CO1.PCCNTR.9382145|3,150,000,000.00|3,150,000.00 |3,146,850,000.00    |99.90               |
|SECRETARIA DISTRITAL DE SEGURIDAD, CONVIVENCIA Y JUSTICIA|CO1.PCCNTR.5147253|4,204,125,000.00|16,837,913.00|4,187,287,087.00    |99.60               |
|Unidad de Busqueda de Personas Desaparecidas             |CO1.PCCNTR.9561451|2,000,000,000.00|17,919,985.00|1,982,080,014.00    |99.10               |
|INSTITUCION UNIVERSITARIA DE BARRANQUILLA-IUB            |CO1.PCCNTR.3628187|10,240,000

## 8. Duración real del contrato (fin − inicio) por modalidad — usa `dim_tiempo` en 2 roles distintos

In [18]:
spark.sql(f"""
SELECT
    m.modalidad_contratacion,
    COUNT(*) AS cantidad_contratos,
    ROUND(AVG(DATEDIFF(t_fin.fecha, t_inicio.fecha)), 1) AS duracion_promedio_dias,
    ROUND(AVG(h.dias_adicionados), 2) AS promedio_dias_adicionados
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_modalidad m     ON h.sk_modalidad = m.sk_modalidad
JOIN {DB_NAME}.dim_tiempo t_inicio ON h.sk_tiempo_inicio = t_inicio.sk_tiempo
JOIN {DB_NAME}.dim_tiempo t_fin    ON h.sk_tiempo_fin    = t_fin.sk_tiempo
WHERE h.dias_adicionados IS NOT NULL
GROUP BY m.modalidad_contratacion
ORDER BY duracion_promedio_dias DESC
""").show(truncate=False)

+-----------------------------------------------------------+------------------+----------------------+-------------------------+
|modalidad_contratacion                                     |cantidad_contratos|duracion_promedio_dias|promedio_dias_adicionados|
+-----------------------------------------------------------+------------------+----------------------+-------------------------+
|Licitación Pública Acuerdo Marco de Precios                |1                 |2248.0                |304.0                    |
|Concurso de méritos abierto                                |38                |543.3                 |85.18                    |
|Licitación pública Obra Publica                            |31                |391.8                 |85.03                    |
|Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes|6                 |291.2                 |10.33                    |
|Licitación pública                                         |34                |285.0     

## 9. Estado de los contratos por trimestre de firma

In [19]:
spark.sql(f"""
SELECT t.anio, t.trimestre, es.estado_contrato, COUNT(*) AS cantidad_contratos
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_tiempo t          ON h.sk_tiempo_firma = t.sk_tiempo
JOIN {DB_NAME}.dim_estado_contrato es ON h.sk_estado = es.sk_estado
GROUP BY t.anio, t.trimestre, es.estado_contrato
ORDER BY t.anio, t.trimestre
""").show(30, truncate=False)

+----+---------+---------------+------------------+
|anio|trimestre|estado_contrato|cantidad_contratos|
+----+---------+---------------+------------------+
|2016|1        |Modificado     |1                 |
|2016|3        |Aprobado       |1                 |
|2016|3        |Cerrado        |3                 |
|2016|4        |Aprobado       |1                 |
|2017|1        |Cerrado        |7                 |
|2017|1        |cedido         |1                 |
|2017|1        |Aprobado       |6                 |
|2017|1        |Prorrogado     |1                 |
|2017|2        |terminado      |5                 |
|2017|2        |Aprobado       |3                 |
|2017|2        |Prorrogado     |1                 |
|2017|3        |terminado      |2                 |
|2017|3        |Aprobado       |5                 |
|2017|3        |Cerrado        |5                 |
|2017|3        |Modificado     |1                 |
|2017|4        |Aprobado       |17                |
|2017|4     

## 10. Categorías de proceso con mayor valor promedio por contrato

In [21]:
spark.sql(f"""
SELECT c.codigo_categoria_principal, c.descripcion_proceso, COUNT(*) AS cantidad_contratos, ROUND(AVG(h.valor_contrato), 2) AS valor_promedio
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_categoria c ON h.sk_categoria = c.sk_categoria
GROUP BY c.codigo_categoria_principal, c.descripcion_proceso
ORDER BY valor_promedio DESC
LIMIT 15
""").show(truncate=False)

+--------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+-----------------+
|codigo_categoria_principal|descripcion_proceso                                                                                                                                                                                                                                                                                         |cantidad_contratos|valor_promedio   |
+--------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Conclusiones

- Este notebook explota el cubo `secop_dw` con 10 consultas analíticas, incluyendo el uso correcto de `dim_tiempo` como dimensión de rol en múltiples combinaciones (firma, inicio+fin simultáneamente).
- Todas combinan `hecho_contratos` con una o más dimensiones, demostrando el valor analítico del modelo.
- Con esto se cierra el flujo completo: **Verificación de entorno → Extracción → Cubo → Transformación → Cargue → Explotación analítica**.